# 01 · Data overview and quality

**Goal:** know what we have before analysing it: which tables, what grain,
what coverage, and what can go wrong.

All data is from [Project CCHAIN](https://data.humdata.org/dataset/project-cchain)
on HDX. Run `python scripts/download.py` from the repo root first.

| Table | Grain | Used for |
|---|---|---|
| `disease_pidsr_totals` | City × week, 2008–2022 | **Target**: leptospirosis cases |
| `climate_atmosphere` | Barangay × day | Rainfall (`pr`, mm) |
| `disease_lgu_disaggregated_totals` | Barangay × week (few cities) | Barangay-level check |
| `project_noah_hazards` | Barangay | Flood hazard (% area) |
| `worldpop_population` | Barangay × year | Population |
| `location` | Barangay | City/barangay names and codes |

In [1]:
from common import *

style()

## 1. The 12 cities

In [2]:
b = barangays()
b.groupby("city").agg(barangays=("adm4_pcode", "size"), area_km2=("brgy_total_area", "sum")).round(1)

,barangays,area_km2
city,,
Cagayan de Oro City,80,506.6
City of Mandaluyong,27,11.3
City of Muntinlupa,9,39.2
City of Navotas,14,10.5
Dagupan City,31,43.2
Davao City,183,2343.7
Iloilo City,180,72.6
Legazpi City,70,157.7
Mandaue City,27,31.1


## 2. Target: PIDSR leptospirosis cases

In [3]:
cases = lepto_weekly()
print(f"rows: {len(cases):,}  cities: {cases.city.nunique()}  "
      f"range: {cases.date.min():%Y-%m-%d} → {cases.date.max():%Y-%m-%d}")
print(f"missing case values: {cases.cases.isna().sum()}  duplicate city-weeks: {cases.duplicated(['city', 'date']).sum()}")
print(f"weekdays of week start: {sorted(int(x) for x in cases.date.dt.day_of_week.unique())}  (0 = Monday)")
cases.groupby("city").size().rename("weeks").to_frame().T

rows: 9,360  cities: 12  range: 2008-01-07 → 2022-12-26
missing case values: 0  duplicate city-weeks: 0
weekdays of week start: [0]  (0 = Monday)


city,Cagayan de Oro City,City of Mandaluyong,City of Muntinlupa,City of Navotas,Dagupan City,Davao City,Iloilo City,Legazpi City,Mandaue City,Palayan City,Tacloban City,Zamboanga City
weeks,780,780,780,780,780,780,780,780,780,780,780,780


Every city has exactly 780 weeks (15 years × 52), with no gaps in values.
But 15 years should span ~783 calendar weeks, so some weeks are missing from the calendar:

In [4]:
gaps = cases[cases.city == ILOILO].assign(gap=lambda d: d.date.diff())
gaps[gaps.gap > pd.Timedelta("7D")][["date", "gap"]]

,date,gap
4940,2013-01-07,14 days
5200,2018-01-08,14 days


**Takeaways**
- Clean: no missing values, no duplicates, weeks start on Monday.
- Two calendar weeks are absent (the weeks of 2012-12-31 and 2018-01-01). PIDSR uses
  52-week years. Lags must be computed **by date, not by row**, or
  these two jumps shift the lag by one week.
- We don't know whether a PIDSR week counts *symptom onset* or *report date*. This
  affects how the rain→cases lag is read (a reporting delay adds to the biological lag).

## 3. Rainfall

In [5]:
rain = rain_daily()
print(f"rows: {len(rain):,}  range: {rain.date.min():%Y-%m-%d} → {rain.date.max():%Y-%m-%d}  "
      f"missing: {rain.pr.isna().sum()}")
rain.groupby("city").pr.describe().round(1)

rows: 87,660  range: 2003-01-01 → 2022-12-31  missing: 0


,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Cagayan de Oro City,7305.0,5.6,7.3,0.0,0.0,2.6,9.1,54.4
City of Mandaluyong,7305.0,6.1,13.1,0.0,0.0,0.0,7.6,186.9
City of Muntinlupa,7305.0,6.7,14.3,0.0,0.0,0.0,8.1,206.7
City of Navotas,7305.0,6.4,13.4,0.0,0.0,0.0,7.9,193.9
Dagupan City,7305.0,6.2,12.7,0.0,0.0,0.0,8.1,173.9
Davao City,7305.0,6.5,8.4,0.0,0.1,3.0,10.2,108.7
Iloilo City,7305.0,6.5,10.8,0.0,0.0,0.0,9.8,129.4
Legazpi City,7305.0,9.8,19.6,0.0,0.0,0.0,11.9,236.6
Mandaue City,7305.0,4.7,7.9,0.0,0.0,0.0,6.9,91.4


In [6]:
p = panel()
print(f"city-weeks with no rain value after joining: {p.rain.isna().sum()}")

city-weeks with no rain value after joining: 0


**Takeaways**
- Rain is a gridded product interpolated to barangays, averaged here to one value per city per day.
- Rain covers the full case period, so every city-week has a rain value.

## 4. Barangay-level case data

In [7]:
lgu = pd.read_csv(RAW / "disease_lgu_disaggregated_totals.csv", parse_dates=["date"])
lgu = lgu[lgu.disease_common_name == "LEPTOSPIROSIS"]
lgu["city"] = lgu.adm3_pcode.map(cities())
lgu.groupby(["city", "source_filename"]).agg(
    rows=("uuid", "size"), barangays=("adm4_pcode", "nunique"), cases=("case_total", "sum"),
    first=("date", "min"), last=("date", "max"),
)

rows  barangays  cases      first       last
city                source_filename                                                                                 
Cagayan de Oro City CdO_2016_INFECTIOUS_DISEASE_clean                     15         11     15 2016-01-04 2016-11-21
                    CdO_2017_INFECTIOUS_DISEASE_clean                      1          1      1 2017-06-05 2017-06-05
                    CdO_2020_INFECTIOUS_DISEASE_clean                     13          9     13 2020-02-24 2020-10-19
                    CdO_2021_INFECTIOUS_DISEASE_clean                      1          1      1 2021-12-13 2021-12-13
                    CdO_2022_INFECTIOUS_DISEASE_clean                     19         15     19 2022-01-03 2022-11-21
                    CdO_2023_Morbidity                                   128          0      0 2023-01-01 2023-08-01
City of Muntinlupa  Muntinlupa_1                                         107          8    110 2010-10-11 2020-09-28
Tacloban City       Tacloban_Infectious_disease_2022                       8          8      8 2022-02-21 2022-11-28
Zamboanga City      Zamboanga_City_Morbidity_Report_Consolidation_2021     2          0      2 2021-11-01 2021-11-01
                    Zamboanga_PIDSR_2013-2022                            383         64    328 2013-01-07 2022-12-26

**Takeaways**
- Only **Zamboanga** (2013–2022, 328 cases across 60 barangays) is usable at barangay level.
  Cagayan de Oro, Muntinlupa and Tacloban have a few dozen cases at most.
- **Iloilo has no barangay case data**, so the barangay ranking can't be validated in the
  city we target. Notebook 04 checks it in Zamboanga instead.

## 5. Hazard and population

In [8]:
noah = pd.read_csv(RAW / "project_noah_hazards.csv")
pop = pd.read_csv(RAW / "worldpop_population.csv", parse_dates=["date"])
print(f"NOAH: {len(noah)} barangays, snapshot {noah.date.unique()}, missing: {noah.isna().sum().sum()}")
print(f"WorldPop: {pop.adm4_pcode.nunique()} barangays, years {pop.date.dt.year.min()}–{pop.date.dt.year.max()}")
noah.filter(like="flood_hazard_25yr").describe().round(1)

NOAH: 879 barangays, snapshot <ArrowStringArray>
['2015-01-01']
Length: 1, dtype: str, missing: 0
WorldPop: 879 barangays, years 2000–2020


,pct_area_flood_hazard_25yr_low,pct_area_flood_hazard_25yr_med,pct_area_flood_hazard_25yr_high
count,879.0,879.0,879.0
mean,6.9,5.7,3.2
std,11.0,12.7,10.0
min,0.0,0.0,0.0
25%,0.0,0.0,0.0
50%,0.0,0.0,0.0
75%,10.6,4.9,1.2
max,69.1,98.0,94.9


**Takeaways**
- NOAH hazard is a **single 2015 snapshot**: share of barangay area in low/medium/high flood
  hazard for 5-, 25- and 100-year rain. It doesn't reflect flood control built since.
- WorldPop stops at **2020**. Later years have to reuse 2020.

## Summary

| Check | Result |
|---|---|
| Case data quality | Clean; 2 missing calendar weeks → lag by date |
| Rain coverage | Complete for 2008–2022 |
| Barangay cases | Only Zamboanga is usable; none for Iloilo |
| Hazard / population | Static 2015 hazard; population to 2020 |